# Imports

In [1]:
from pathlib import Path
print(Path.cwd())

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent)) # problem with dependency resolution (e.g. custom_builder) without this

/Users/mac/Documents/dev/ID2221/dic/Week 2


In [2]:
# Use delta features if needed (DeltaTable, etc.)
from delta import *
from custom_builder import builder
from log import *
import numpy as np
from Queries import *

# use the existing preconfigured builder to create the Spark session.
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f'current database: {spark.catalog.currentDatabase()}')
print(f'spark tables: {spark.catalog.listTables()}')

from pyspark.sql import functions as F

import json

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/16 15:29:09 WARN Utils: Your hostname, MacBook-Pro-som-tillhor-MAC.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.247 instead (on interface en0)
26/09/16 15:29:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/mac/Documents/dev/ID2221/dic/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/mac/.ivy2.5.2/cache
The jars for the packages stored in: /Users/mac/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b5f96d62-e5c4-4a46-9d2c-b1fe04044b45;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-ap

current database: default
spark tables: [Table(name='air_quality', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='integrated_taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_zone_lookup', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='weather', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False)]


# Ensure uncached tables

In [3]:
spark.catalog.uncacheTable("default.air_quality")
spark.catalog.uncacheTable("default.taxi_trips")
spark.catalog.uncacheTable("default.taxi_zone_lookup")
spark.catalog.uncacheTable("default.weather")

# Ensure no auto broadcast

In [4]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

# reset the broadcast threshold to default value
# spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)  # 10 MB

# Regular queries
Make sure to manually check no broadcast occurs in execution plan

In [17]:
result = spark.sql(query_2_1(broadcast=False))
result.show()
result.explain(extended=True)

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|            Kips Bay|    1|    32984|
|                SoHo|    1|    20197|
|Upper East Side N...|   12|        1|
|Breezy Point/Fort...|    1|        6|
|    Bensonhurst West|    1|      153|
|       East New York|    1|      938|
|Flushing Meadows-...|    1|      510|
|        Battery Park|    1|      875|
|        Borough Park|    1|      237|
| Grymes Hill/Clifton|    1|        2|
|          Kensington|    1|      127|
|         Hunts Point|    1|      105|
|       Melrose South|    1|      289|
|       Prospect Park|    1|       53|
|          Pelham Bay|    1|       47|
| UN/Turtle Bay South|    1|    33595|
|Washington Height...|    1|      495|
|       Willets Point|    1|       14|
|Downtown Brooklyn...|    1|     1393|
|    Inwood Hill Park|    1|       22|
+--------------------+-----+---------+
only showing top 20 rows
== Parsed Logical Plan ==
'Aggregate ['

In [18]:
result = spark.sql(query_2_2(broadcast=False))
result.show()
result.explain(extended=True)

+--------------+-------+------------------+
|  column_group|    cnt| avg_trip_distance|
+--------------+-------+------------------+
|greater_than_0| 424773| 3.470743355446981|
|  zero_or_null|2539795|3.6824345463125323|
+--------------+-------+------------------+

== Parsed Logical Plan ==
'Aggregate [CASE WHEN ('prcp > 0) THEN greater_than_0 ELSE zero_or_null END], [CASE WHEN ('prcp > 0) THEN greater_than_0 ELSE zero_or_null END AS column_group#7189, 'COUNT(1) AS cnt#7190, 'AVG('trip_distance) AS avg_trip_distance#7191]
+- 'Join LeftOuter, ('date_trunc(hour, 't.pu_datetime) = 'w.datetime)
   :- 'SubqueryAlias t
   :  +- 'UnresolvedRelation [default, taxi_trips], [], false
   +- 'SubqueryAlias w
      +- 'UnresolvedRelation [default, weather], [], false

== Analyzed Logical Plan ==
column_group: string, cnt: bigint, avg_trip_distance: double
Aggregate [CASE WHEN (cast(prcp#7201 as double) > cast(0 as double)) THEN greater_than_0 ELSE zero_or_null END], [CASE WHEN (cast(prcp#7201 as dou

In [19]:
result = spark.sql(query_2_3(broadcast=False))
result.show()
result.explain(extended=True)

+-----------+-------+
|measurement|  trips|
+-----------+-------+
|       NULL|2945398|
|        1.3|     20|
|        1.6|     13|
|        1.7|     31|
|        1.8|     41|
|        1.9|     19|
|        2.0|     24|
|        2.1|    135|
|        2.2|     39|
|        2.3|     36|
|        2.4|     51|
|        2.5|    137|
|        2.6|     93|
|        2.7|     84|
|        2.8|     27|
|        2.9|    141|
|        3.0|     62|
|        3.1|     89|
|        3.2|    115|
|        3.3|     71|
+-----------+-------+
only showing top 20 rows
== Parsed Logical Plan ==
'Sort ['measurement ASC NULLS FIRST, 'trips ASC NULLS FIRST], true
+- 'Aggregate ['measurement], ['measurement, 'COUNT('county) AS trips#7534]
   +- 'Join LeftOuter, (('date_trunc(hour, 'pu_datetime) = 'hr_datetime) AND ('county = 'aq_county))
      :- 'Join LeftOuter, ('t.pu_location_id = 'tzl.location_id)
      :  :- 'SubqueryAlias t
      :  :  +- 'UnresolvedRelation [taxi_trips], [], false
      :  +- 'SubqueryAli

# Broadcasted queries
Make sure to manually check broadcast occurs in execution plan

In [29]:
result = spark.sql(query_2_1(broadcast=True))
result.show()
result.explain(extended=True)

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|  Van Cortlandt Park|    1|       18|
|                SoHo|    1|    20197|
|            Kips Bay|    1|    32984|
|Upper West Side S...|    1|    88474|
|Upper East Side S...|    1|   142708|
|       Rockaway Park|    1|       80|
|           Stapleton|    1|        4|
|East New York/Pen...|    1|      245|
|          Bath Beach|    1|       58|
|  Claremont/Bathgate|    1|      173|
|Bay Terrace/Fort ...|    1|       38|
|       Fordham South|    1|       91|
|             Bayside|    1|       81|
|    Garment District|    1|    48093|
|     Cambria Heights|    1|      143|
|    Bensonhurst East|    1|      145|
|         Great Kills|    1|        1|
|Upper West Side N...|    1|    64234|
|   Kew Gardens Hills|    1|      151|
|Springfield Garde...|    1|      446|
+--------------------+-----+---------+
only showing top 20 rows
== Parsed Logical Plan ==
'UnresolvedHi

In [30]:
result = spark.sql(query_2_2(broadcast=True))
result.show()
result.explain(extended=True)

+--------------+-------+------------------+
|  column_group|    cnt| avg_trip_distance|
+--------------+-------+------------------+
|greater_than_0| 424773| 3.470743355446981|
|  zero_or_null|2539795|3.6824345463125323|
+--------------+-------+------------------+

== Parsed Logical Plan ==
'UnresolvedHint BROADCAST, ['w]
+- 'Aggregate [CASE WHEN ('prcp > 0) THEN greater_than_0 ELSE zero_or_null END], [CASE WHEN ('prcp > 0) THEN greater_than_0 ELSE zero_or_null END AS column_group#11806, 'COUNT(1) AS cnt#11807, 'AVG('trip_distance) AS avg_trip_distance#11808]
   +- 'Join LeftOuter, ('date_trunc(hour, 't.pu_datetime) = 'w.datetime)
      :- 'SubqueryAlias t
      :  +- 'UnresolvedRelation [default, taxi_trips], [], false
      +- 'SubqueryAlias w
         +- 'UnresolvedRelation [default, weather], [], false

== Analyzed Logical Plan ==
column_group: string, cnt: bigint, avg_trip_distance: double
Aggregate [CASE WHEN (cast(prcp#11818 as double) > cast(0 as double)) THEN greater_than_0 ELS

In [31]:
result = spark.sql(query_2_3(broadcast=True))
result.show()
result.explain(extended=True)

+-----------+-------+
|measurement|  trips|
+-----------+-------+
|       NULL|2945398|
|        1.3|     20|
|        1.6|     13|
|        1.7|     31|
|        1.8|     41|
|        1.9|     19|
|        2.0|     24|
|        2.1|    135|
|        2.2|     39|
|        2.3|     36|
|        2.4|     51|
|        2.5|    137|
|        2.6|     93|
|        2.7|     84|
|        2.8|     27|
|        2.9|    141|
|        3.0|     62|
|        3.1|     89|
|        3.2|    115|
|        3.3|     71|
+-----------+-------+
only showing top 20 rows
== Parsed Logical Plan ==
'Sort ['measurement ASC NULLS FIRST, 'trips ASC NULLS FIRST], true
+- 'UnresolvedHint BROADCAST, ['tzl]
   +- 'UnresolvedHint BROADCAST, ['aq]
      +- 'Aggregate ['measurement], ['measurement, 'COUNT('county) AS trips#12151]
         +- 'Join LeftOuter, (('date_trunc(hour, 'pu_datetime) = 'hr_datetime) AND ('county = 'aq_county))
            :- 'Join LeftOuter, ('t.pu_location_id = 'tzl.location_id)
            :  :-